In [1]:
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import os
from scipy.spatial import distance_matrix

# --- Configuration ---
"""TARGET_GENES = [
    'Eno1', 'Mapt', 'Thy1', 'Pmch', 'Atp1a3', 'Rac1', 'Rsrp1', 'Snhg11', 'Tubb4a',
    'Rasgrf1', 'Hsp90ab1', 'Elavl3', 'App', 'Syp', 'AC149090.1',
    'Aplp1', 'Apoe', 'Meg3', 'Gnas', 'Pkm'
]"""
TARGET_GENES = ['App',
                'Mapt',
                'Apoe',
                'C1qa',
                'C1qb',
                'C1qc',
                'Gfap',
                'S100b',
                'Trem2',
                'Tyrobp',
                'Itgam',
                'Aif1',
                'Syn1',
                'Syp',
                'Dlg4',
                'Shank2',
                'Nrcam',
                'Baiap2',
                'Cox6a1',
                'Cox7a2',
                'Ndufa1',
                'Ndufb6',
                'Uqcrb',
                'Uqcr10',
                'Mbp',
                'Plp1',
                'Mog',
                'Hspa1a',
                'Hsp90ab1',
                'Sod2']
"""['App', 'Psen1', 'Psen2', 'Mapt', 'Apoe', 'Trem2', 'Tyrobp',
 'C1qa', 'C1qb', 'C1qc', 'Csf1r', 'Aif1', 'Cst7',
 'Sele', 'Ccr6', 'Aqp4', 'Plcg2', 'Abca7', 'Bace1']"""

#RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_all/"

RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]

RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

SPOT_DIAMETER_UM = 55
SPOT_SPACING_UM = 100 
PERCENTILE_VMAX = 100  
USE_RAW_DATA = False 
FLIP_HORIZONTAL = True  
OUTPUT_DIR = "/home/ajarrah/PhD_Thesis/chapter_4/code_final/gene_plots_aged/"

# Custom Colormap
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),         # dark gray 
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),        # navy
    (0.15, "#0000FF"),        # blue
    (0.30, "#8000FF"),        # purple-ish
    (0.45, "#FF0000"),        # red
    (0.60, "#FF8000"),        # orange
    (0.75, "#FFFF00"),        # yellow
    (1.0, "#FFFFFF")          # white
])

def calculate_spot_size(adata, spot_diameter_um=55, spot_spacing_um=100):
    if "spatial" not in adata.obsm: return None
    coords = adata.obsm["spatial"]
    sample_coords = coords[np.random.choice(coords.shape[0], min(1000, coords.shape[0]), replace=False)]
    dist_mat = distance_matrix(sample_coords, sample_coords)
    np.fill_diagonal(dist_mat, np.inf)
    scale = np.median(np.min(dist_mat, axis=1)) / spot_spacing_um
    return spot_diameter_um * scale

def find_gene_in_adata(adata, gene_name):
    if gene_name in adata.var_names: return gene_name
    for var in adata.var_names:
        if var.lower() == gene_name.lower(): return var
    return None

def plot_gene(gene_name, output_dir=OUTPUT_DIR, use_raw=USE_RAW_DATA, 
              percentile_vmax=PERCENTILE_VMAX, flip_horizontal=FLIP_HORIZONTAL):
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(4, 4, figsize=(20, 16))
    axes = axes.flatten()
    data_type = "Raw" if use_raw else "Norm"
    
    print(f"\nPlotting {gene_name} (Local Scaling 0-p{percentile_vmax})")
    
    for i, (filename, sample_id) in enumerate(zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS)):
        ax = axes[i]
        file_path = os.path.join(RNA_INPUT_FOLDER, filename)
        
        try:
            adata = sc.read_h5ad(file_path)
            # CRITICAL: Fix non-unique var names to prevent indexing errors
            if not adata.var_names.is_unique:
                adata.var_names_make_unique()
            
            gene_to_plot = find_gene_in_adata(adata, gene_name)
            if gene_to_plot is None:
                ax.set_title(f"{sample_id}\nNot Found")
                continue

            # Data selection
            if use_raw and hasattr(adata, 'raw') and adata.raw is not None:
                expr = adata.raw[:, gene_to_plot].X
            else:
                expr = adata[:, gene_to_plot].X
            
            expr = expr.toarray().flatten() if hasattr(expr, 'toarray') else expr.flatten()
            
            # LOCAL VMAX CALCULATION
            local_vmax = np.percentile(expr, percentile_vmax)
            
            if flip_horizontal:
                adata.obsm['spatial'][:, 0] = -1 * adata.obsm['spatial'][:, 0]

            sc.pl.spatial(
                adata, color=gene_to_plot, ax=ax, show=False, 
                title=f"{sample_id}\nmax={local_vmax:.1f}",
                frameon=False, spot_size=calculate_spot_size(adata, SPOT_DIAMETER_UM, SPOT_SPACING_UM),
                cmap=custom_cmap, vmin=0, vmax=local_vmax, use_raw=use_raw
            )
            print(f"  {sample_id}: Done", end="\r")

        except Exception as e:
            print(f"  {sample_id}: Error {str(e)[:20]}")
            ax.set_title(f"{sample_id} Error")

    plt.tight_layout()
    out_path = os.path.join(output_dir, f"{gene_name}_{data_type}_local_p{percentile_vmax}.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nSaved: {out_path}")

def plot_all_genes():
    for gene in TARGET_GENES:
        plot_gene(gene)

if __name__ == "__main__":
    plot_all_genes()


Plotting App (Local Scaling 0-p100)


/tmp/ipykernel_313374/2170088407.py:139: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/2170088407.py:139: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/2170088407.py:139: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/2170088407.py:139: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


...

/tmp/ipykernel_313374/2170088407.py:139: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/2170088407.py:139: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/2170088407.py:139: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/2170088407.py:139: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


  AAD_4: Done
Saved: /home/ajarrah/PhD_Thesis/chapter_4/code_final/gene_plots_aged/Sod2_Norm_local_p100.png


In [2]:
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import os
from scipy.spatial import distance_matrix

# --- Configuration ---
"""TARGET_GENES = [
    'Eno1', 'Mapt', 'Thy1', 'Pmch', 'Atp1a3', 'Rac1', 'Rsrp1', 'Snhg11', 'Tubb4a',
    'Rasgrf1', 'Hsp90ab1', 'Elavl3', 'App', 'Syp', 'AC149090.1',
    'Aplp1', 'Apoe', 'Meg3', 'Gnas', 'Pkm'
]"""

TARGET_GENES = ['App',
                'Mapt',
                'Apoe',
                'C1qa',
                'C1qb',
                'C1qc',
                'Gfap',
                'S100b',
                'Trem2',
                'Tyrobp',
                'Itgam',
                'Aif1',
                'Syn1',
                'Syp',
                'Dlg4',
                'Shank2',
                'Nrcam',
                'Baiap2',
                'Cox6a1',
                'Cox7a2',
                'Ndufa1',
                'Ndufb6',
                'Uqcrb',
                'Uqcr10',
                'Mbp',
                'Plp1',
                'Mog',
                'Hspa1a',
                'Hsp90ab1',
                'Sod2']
"""['App', 'Psen1', 'Psen2', 'Mapt', 'Apoe', 'Trem2', 'Tyrobp',
 'C1qa', 'C1qb', 'C1qc', 'Csf1r', 'Aif1', 'Cst7',
 'Sele', 'Ccr6', 'Aqp4', 'Plcg2', 'Abca7', 'Bace1']"""

#RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_all/"

RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]

RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

SPOT_DIAMETER_UM = 100
SPOT_SPACING_UM = 100 
PERCENTILE_VMAX = 100  
USE_RAW_DATA = False 
FLIP_HORIZONTAL = True  
OUTPUT_DIR = "/home/ajarrah/PhD_Thesis/chapter_4/code_final/gene_plots_biggerspots_aged/"

# Custom Colormap
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),         # dark gray 
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),        # navy
    (0.15, "#0000FF"),        # blue
    (0.30, "#8000FF"),        # purple-ish
    (0.45, "#FF0000"),        # red
    (0.60, "#FF8000"),        # orange
    (0.75, "#FFFF00"),        # yellow
    (1.0, "#FFFFFF")          # white
])

def calculate_spot_size(adata, spot_diameter_um=55, spot_spacing_um=100):
    if "spatial" not in adata.obsm: return None
    coords = adata.obsm["spatial"]
    sample_coords = coords[np.random.choice(coords.shape[0], min(1000, coords.shape[0]), replace=False)]
    dist_mat = distance_matrix(sample_coords, sample_coords)
    np.fill_diagonal(dist_mat, np.inf)
    scale = np.median(np.min(dist_mat, axis=1)) / spot_spacing_um
    return spot_diameter_um * scale

def find_gene_in_adata(adata, gene_name):
    if gene_name in adata.var_names: return gene_name
    for var in adata.var_names:
        if var.lower() == gene_name.lower(): return var
    return None

def plot_gene(gene_name, output_dir=OUTPUT_DIR, use_raw=USE_RAW_DATA, 
              percentile_vmax=PERCENTILE_VMAX, flip_horizontal=FLIP_HORIZONTAL):
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(4, 4, figsize=(20, 16))
    axes = axes.flatten()
    data_type = "Raw" if use_raw else "Norm"
    
    print(f"\nPlotting {gene_name} (Local Scaling 0-p{percentile_vmax})")
    
    for i, (filename, sample_id) in enumerate(zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS)):
        ax = axes[i]
        file_path = os.path.join(RNA_INPUT_FOLDER, filename)
        
        try:
            adata = sc.read_h5ad(file_path)
            # CRITICAL: Fix non-unique var names to prevent indexing errors
            if not adata.var_names.is_unique:
                adata.var_names_make_unique()
            
            gene_to_plot = find_gene_in_adata(adata, gene_name)
            if gene_to_plot is None:
                ax.set_title(f"{sample_id}\nNot Found")
                continue

            # Data selection
            if use_raw and hasattr(adata, 'raw') and adata.raw is not None:
                expr = adata.raw[:, gene_to_plot].X
            else:
                expr = adata[:, gene_to_plot].X
            
            expr = expr.toarray().flatten() if hasattr(expr, 'toarray') else expr.flatten()
            
            # LOCAL VMAX CALCULATION
            local_vmax = np.percentile(expr, percentile_vmax)
            
            if flip_horizontal:
                adata.obsm['spatial'][:, 0] = -1 * adata.obsm['spatial'][:, 0]

            sc.pl.spatial(
                adata, color=gene_to_plot, ax=ax, show=False, 
                title=f"{sample_id}\nmax={local_vmax:.1f}",
                frameon=False, spot_size=calculate_spot_size(adata, SPOT_DIAMETER_UM, SPOT_SPACING_UM),
                cmap=custom_cmap, vmin=0, vmax=local_vmax, use_raw=use_raw
            )
            print(f"  {sample_id}: Done", end="\r")

        except Exception as e:
            print(f"  {sample_id}: Error {str(e)[:20]}")
            ax.set_title(f"{sample_id} Error")

    plt.tight_layout()
    out_path = os.path.join(output_dir, f"{gene_name}_{data_type}_local_p{percentile_vmax}.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\nSaved: {out_path}")

def plot_all_genes():
    for gene in TARGET_GENES:
        plot_gene(gene)

if __name__ == "__main__":
    plot_all_genes()


Plotting App (Local Scaling 0-p100)


/tmp/ipykernel_313374/3062891863.py:140: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/3062891863.py:140: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/3062891863.py:140: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/3062891863.py:140: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


...

/tmp/ipykernel_313374/3062891863.py:140: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/3062891863.py:140: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/3062891863.py:140: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


/tmp/ipykernel_313374/3062891863.py:140: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


  AAD_4: Done
Saved: /home/ajarrah/PhD_Thesis/chapter_4/code_final/gene_plots_biggerspots_aged/Sod2_Norm_local_p100.png
